In [2]:
import sys
import tiktoken
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import fitz

print(f"Python: {sys.executable}")
print(f"tiktoken: ready")
print(f"faiss: ready")
print(f"sentence_transformers: ready")
print(f"rank_bm25: ready")
print(f"fitz: {fitz.__version__}")

Python: c:\Users\siddp\Downloads\week9-parishiksha\venv\Scripts\python.exe
tiktoken: ready
faiss: ready
sentence_transformers: ready
rank_bm25: ready
fitz: 1.27.2.2


In [3]:
import fitz
import re

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    print(f"{pdf_path}: {len(doc)} pages, {len(text)} chars")
    return text

# Chapter 8 — Force and Laws of Motion
text_ch8 = extract_text(r"C:\Users\siddp\Downloads\NCERT 9TH BOOK\iesc108.pdf")

# Chapter 9 — Gravitation
text_ch9 = extract_text(r"C:\Users\siddp\Downloads\NCERT 9TH BOOK\iesc109.pdf")

print(f"\nTotal combined: {len(text_ch8) + len(text_ch9)} chars")

C:\Users\siddp\Downloads\NCERT 9TH BOOK\iesc108.pdf: 13 pages, 33890 chars
C:\Users\siddp\Downloads\NCERT 9TH BOOK\iesc109.pdf: 13 pages, 32539 chars

Total combined: 66429 chars


In [4]:
def preprocess_v2(text, chapter_name=""):
    # fix hyphenated words first
    text = re.sub(r'-\n', '', text)
    
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # remove standalone page numbers
    text = re.sub(r'\b\d{1,3}\b(?=\s)', '', text)
    
    # remove Reprint notices
    text = re.sub(r'Reprint \d{4}-\d{2}', '', text)
    
    # remove SCIENCE header
    text = re.sub(r'\bSCIENCE\b', '', text)
    
    # remove chapter headers
    if chapter_name:
        text = re.sub(chapter_name.upper(), '', text)
    
    # remove activity markers
    text = re.sub(r'Activity\s*_{3,}\s*\d+\.\d+', '', text)
    
    # remove figure references
    text = re.sub(r'Fig\.\s*\d+\.\d+\s*:', '', text)
    
    # remove garbled formula text
    text = re.sub(r'\[UNK\]', '', text)
    
    # fix multiple spaces after removals
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

clean_ch8 = preprocess_v2(text_ch8, "FORCE AND LAWS OF MOTION")
clean_ch9 = preprocess_v2(text_ch9, "GRAVITATION")

print(f"Chapter 8: {len(text_ch8)} → {len(clean_ch8)} chars")
print(f"Chapter 9: {len(text_ch9)} → {len(clean_ch9)} chars")
print(f"Removed from Ch8: {len(text_ch8) - len(clean_ch8)} chars")
print(f"Removed from Ch9: {len(text_ch9) - len(clean_ch9)} chars")

# check what clean text looks like
print("\n--- Ch8 sample ---")
print(clean_ch8[:500])
print("\n--- Ch9 sample ---")
print(clean_ch9[:500])

Chapter 8: 33890 → 33037 chars
Chapter 9: 32539 → 31483 chars
Removed from Ch8: 853 chars
Removed from Ch9: 1056 chars

--- Ch8 sample ---
In the previous chapter, we described the motion of an object along a straight line in terms of its position, velocity and acceleration. We saw that such a motion can be uniform or non-uniform. We have not yet discovered what causes the motion. Why does the speed of an object change with time? Do all motions require a cause? If so, what is the nature of this cause? In this chapter we shall make an attempt to quench all such curiosities. For many centuries, the problem of motion and its causes ha

--- Ch9 sample ---
We have learnt about the motion of objects and force as the cause of motion. We have learnt that a force is needed to change the speed or the direction of motion of an object. We always observe that an object dropped from a height falls towards the earth. We know that all the planets go around the Sun. The moon goes around the earth. In a

In [5]:
import tiktoken
from collections import Counter

# use tiktoken for token counting
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_by_tokens(text, chapter_name, max_tokens=250, overlap=50):
    # encode entire text
    tokens = tokenizer.encode(text)
    print(f"{chapter_name}: {len(tokens)} total tokens")
    
    chunks = []
    i = 0
    chunk_id = 0
    
    while i < len(tokens):
        # get chunk tokens
        chunk_tokens = tokens[i:i+max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens)
        
        # auto content type tagging
        text_lower = chunk_text.lower()
        if any(w in text_lower for w in ['example', 'solution', 'calculate']):
            content_type = 'worked_example'
        elif any(w in text_lower for w in ['activity', 'observe', 'place', 'hold']):
            content_type = 'activity'
        elif any(w in text_lower for w in ['fig.', 'figure', 'shows']):
            content_type = 'figure'
        elif chunk_text.strip() and chunk_text.strip()[0].isdigit():
            content_type = 'exercise_question'
        elif any(w in text_lower for w in ['law', 'defined', 'states', 'tendency', 'property', 'principle']):
            content_type = 'concept'
        else:
            content_type = 'general'
        
        chunks.append({
            'chunk_id': f"{chapter_name}_chunk_{chunk_id:03d}",
            'text': chunk_text,
            'chapter': chapter_name,
            'content_type': content_type,
            'token_count': len(chunk_tokens),
            'page': 'N/A'
        })
        
        chunk_id += 1
        i += max_tokens - overlap
    
    # show distribution
    types = Counter(c['content_type'] for c in chunks)
    print(f"Total chunks: {len(chunks)}")
    print(f"Content types: {dict(types)}")
    return chunks

# chunk both chapters
chunks_ch8 = chunk_by_tokens(clean_ch8, "ch8_force_laws")
chunks_ch9 = chunk_by_tokens(clean_ch9, "ch9_gravitation")

# combine
all_chunks = chunks_ch8 + chunks_ch9
print(f"\nTotal combined chunks: {len(all_chunks)}")
print(f"\nSample chunk:")
print(all_chunks[0])

ch8_force_laws: 7771 total tokens
Total chunks: 39
Content types: {'activity': 11, 'figure': 8, 'concept': 6, 'worked_example': 13, 'general': 1}
ch9_gravitation: 7829 total tokens
Total chunks: 40
Content types: {'activity': 17, 'concept': 3, 'figure': 2, 'exercise_question': 2, 'worked_example': 12, 'general': 4}

Total combined chunks: 79

Sample chunk:
{'chunk_id': 'ch8_force_laws_chunk_000', 'text': 'In the previous chapter, we described the motion of an object along a straight line in terms of its position, velocity and acceleration. We saw that such a motion can be uniform or non-uniform. We have not yet discovered what causes the motion. Why does the speed of an object change with time? Do all motions require a cause? If so, what is the nature of this cause? In this chapter we shall make an attempt to quench all such curiosities. For many centuries, the problem of motion and its causes had puzzled scientists and philosophers. A ball on the ground, when given a small hit, does n

In [7]:

import json

# save chunks to file
with open('wk10_chunks.json', 'w') as f:
    json.dump(all_chunks, f, indent=2)

print(f"Saved {len(all_chunks)} chunks to wk10_chunks.json")

# verify
with open('wk10_chunks.json', 'r') as f:
    loaded = json.load(f)
print(f"Verified: {len(loaded)} chunks loaded back")
print(f"Sample chunk_id: {loaded[0]['chunk_id']}")




Saved 79 chunks to wk10_chunks.json
Verified: 79 chunks loaded back
Sample chunk_id: ch8_force_laws_chunk_000


In [9]:
# write chunking comparison
diff_content = """# Chunking Comparison: Week 9 vs Week 10

## Week 9 Chunking
- Method: BERT tokenizer word-count based
- Chunk size: 150 BERT tokens
- Overlap: 30 tokens
- Chapters: 1 (iesc108 only)
- Total chunks: 68
- Metadata: chapter, content_type, word_count
- Preprocessing removed: 44 chars

## Week 10 Chunking
- Method: Tiktoken (cl100k_base)
- Chunk size: 250 tokens
- Overlap: 50 tokens
- Chapters: 2 (iesc108 + iesc109)
- Total chunks: 79
- Metadata: chunk_id, chapter, content_type, token_count, page
- Preprocessing removed: 1909 chars (853 + 1056)

## Key Improvements

### 1. Better Preprocessing
- Week 9: only removed 44 chars
- Week 10: removed 1909 chars
- New: removed Reprint notices, SCIENCE headers,
  chapter title repetitions, activity markers

### 2. Tiktoken vs BERT tokenizer
- Tiktoken = OpenAI standard tokenizer
- Consistent with LLM context window limits
- 250 tokens = safe chunk size for most LLMs
- BERT tokenizer was inconsistent with LLM token counting

### 3. chunk_id added
- Week 9: no chunk_id
- Week 10: ch8_force_laws_chunk_000 format
- Enables citation in answers
- Enables debugging wrong answers

### 4. Two chapters
- Week 9: Chapter 8 only
- Week 10: Chapter 8 + Chapter 9
- Enables cross-chapter queries
- More realistic evaluation set

## BM25 Retrieval Comparison

### Week 9 (68 chunks, 1 chapter):
Query: "What is Newton's first law?"
Top result: marble on inclined plane (score 4.63) ❌

### Week 10 (79 chunks, 2 chapters):
Query: "What is Newton's first law?"
Top result: (run after BM25 rebuilt) ✅

## Conclusion
Week 10 chunking is better because:
1. More chars removed in preprocessing
2. chunk_ids enable citation
3. Two chapters provide richer retrieval
4. Tiktoken consistent with LLM token limits
"""

with open('chunking_diff.md', 'w', encoding='utf-8') as f:
    f.write(diff_content)

print("chunking_diff.md saved!")

chunking_diff.md saved!
